# Лабораторная 1. Введение в Apache Spark

## Подготовка среды в Google Colab

Установка зависимостей

In [1]:
!apt-get update -qq > /dev/null 2>&1
!apt-get install openjdk-11-jdk-headless -y -qq > /dev/null 2>&1
!wget --show-progress -q http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz -O spark.tgz
!tar xf spark.tgz -C /content/ > /dev/null
!pip install -q findspark

spark.tgz           100%[===================>] 381.90M   488KB/s    in 23m 48s 


Настройка переменных окружения

In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

Инициализация Spark

In [3]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
# Создаем сессию Spark
spark = SparkSession.builder.master("local[*]").getOrCreate()
# Получаем доступ к контексту Spark (sc), который нужен для RDD
sc = spark.sparkContext
spark

## Создание Resilient Distributed Dataset

Загрузка текстового файла warandpeace.txt

In [4]:
!wget -O warandsociety.txt https://git.ai.ssau.ru/tk/big_data/raw/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/warandsociety.txt

--2026-03-12 18:46:47--  https://git.ai.ssau.ru/tk/big_data/raw/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/warandsociety.txt
Resolving git.ai.ssau.ru (git.ai.ssau.ru)... 91.222.131.161
Connecting to git.ai.ssau.ru (git.ai.ssau.ru)|91.222.131.161|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: /tk/big_data/raw/commit/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/warandsociety.txt [following]
--2026-03-12 18:46:48--  https://git.ai.ssau.ru/tk/big_data/raw/commit/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/warandsociety.txt
Reusing existing connection to git.ai.ssau.ru:443.
HTTP request sent, awaiting response... 200 OK
Length: 5315699 (5.1M) [text/plain]
Saving to: ‘warandsociety.txt’

warandsociety.txt   100%[===================>]   5.07M  3.52MB/s    in 1.4s    

2026-03-12 18:46:50 (3.52 MB/s) - ‘warandsociety.txt’ saved [5315699/5315699]



Создайте RDD для текстового файла warandpeace.txt

In [5]:
warandpeace = sc.textFile("warandsociety.txt")

Выведите количество строк файла

In [6]:
warandpeace.count()

12851

Попробуйте считать несуществующий файл, например `nil`, а затем вывести количество его строк на экран

In [7]:
nilFile = sc.textFile("nil")
try:
    print(nilFile.count())
except Exception as e:
    print("Файл не найден (ошибка ожидаема):", e)

Файл не найден (ошибка ожидаема): An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.hadoop.mapred.InvalidInputException: Input path does not exist: file:/content/nil
	at org.apache.hadoop.mapred.FileInputFormat.singleThreadedListStatus(FileInputFormat.java:304)
	at org.apache.hadoop.mapred.FileInputFormat.listStatus(FileInputFormat.java:244)
	at org.apache.hadoop.mapred.FileInputFormat.getSplits(FileInputFormat.java:332)
	at org.apache.spark.rdd.HadoopRDD.getPartitions(HadoopRDD.scala:208)
	at org.apache.spark.rdd.RDD.$anonfun$partitions$2(RDD.scala:294)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.rdd.RDD.partitions(RDD.scala:290)
	at org.apache.spark.rdd.MapPartitionsRDD.getPartitions(MapPartitionsRDD.scala:49)
	at org.apache.spark.rdd.RDD.$anonfun$partitions$2(RDD.scala:294)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.rdd.RDD.partitions(RDD.scala:290)
	at org.apache.spark.api.python.Python

Заметьте, что первая команда выполняется успешно, а вторая выводит сообщение, что такого файла нет. Это происходит потому, что выполнение обработки в Spark является ленивым и не запускается, до встречи команды действия(action). `count` - первая команда действия, с которой вы познакомились.

Считайте первые 10 строк файла warandsociety.txt

In [8]:
warandpeace.take(10)

['Лев Николаевич Толстой',
 'Война и мир. Книга 1',
 '',
 'Война и мир – 1',
 '',
 ' ',
 ' http://www.lib.ru',
 '',
 'Аннотация ',
 '']

Эта команда не требует считывания и передачи на главный узел всех данных RDD.

Узнайте на сколько частей разделились данные в кластере

In [9]:
warandpeace.getNumPartitions()

2

Создайте распределённую коллекцию из нескольких элементов и для каждого элемента верните ip адрес, на котором он расположен:

In [10]:
import socket

sc.parallelize([1, 2, 3]).map(
    lambda x: socket.gethostbyname(socket.gethostname())
).collect()

['172.28.0.12', '172.28.0.12', '172.28.0.12']

## Обработка текста

Найдите строки, в которых содержится слово "война"

In [11]:
linesWithWar = warandpeace.filter(lambda x: "война" in x)

Запросите первую строку. Строкой в данном файле является целый абзац, так как только по завершению абзаца содержится символ переноса строки.

In [12]:
linesWithWar.first()

"– Еh bien, mon prince. Genes et Lucques ne sont plus que des apanages, des поместья, de la famille Buonaparte. Non, je vous previens, que si vous ne me dites pas, que nous avons la guerre, si vous vous permettez encore de pallier toutes les infamies, toutes les atrocites de cet Antichrist (ma parole, j'y crois) – je ne vous connais plus, vous n'etes plus mon ami, vous n'etes plus мой верный раб, comme vous dites. [Ну, что, князь, Генуа и Лукка стали не больше, как поместьями фамилии Бонапарте. Нет, я вас предупреждаю, если вы мне не скажете, что у нас война, если вы еще позволите себе защищать все гадости, все ужасы этого Антихриста (право, я верю, что он Антихрист) – я вас больше не знаю, вы уж не друг мой, вы уж не мой верный раб, как вы говорите.] Ну, здравствуйте, здравствуйте. Je vois que je vous fais peur, [Я вижу, что я вас пугаю,] садитесь и рассказывайте."

Данные могут быть перемещены в кэш. Этот приём очень полезен при повторном обращении к данным, для запросов "горячих" данных или запуска итеративных алгоритмов.

Перед подсчётом количества элементов вызовите команду кэширования `cache()`. Трансформации не будут обработаны, пока не будет запущена одна из команд - действий.

In [13]:
linesWithWar.cache()

PythonRDD[10] at RDD at PythonRDD.scala:53

Первый запуск (медленнее, т.к. вычисляется)

In [14]:
%%time
linesWithWar.count()

CPU times: user 8.8 ms, sys: 3.25 ms, total: 12 ms
Wall time: 1.29 s


54

Второй запуск (быстрее, т.к. в кэше)

In [15]:
%%time
linesWithWar.count()

CPU times: user 5.36 ms, sys: 833 µs, total: 6.19 ms
Wall time: 339 ms


54

При выполнении команды count второй раз вы должны заметить небольшое ускорение. Кэширование небольших файлов не даёт большого преимущества, однако для огромных файлов, распределённых по сотням или тысячам узлов, разница во времени выполнения может быть существенной. Вторая команда `linesWithWar.count()` выполняется над результатом от предшествующих команде cache трансформаций и на больших объёмах данных будет ускорять выполнение последующих команд.

Найдите гистограмму слов:

In [16]:
wordCounts = linesWithWar.flatMap(lambda line: line.split(" ")) \
                         .map(lambda word: (word, 1)) \
                         .reduceByKey(lambda a, b: a + b)

wordCounts.saveAsTextFile("warandpeace_histogram.txt")
!head warandpeace_histogram.txt/part-00000

('Еh', 1)
('prince.', 1)
('et', 4)
('sont', 1)
('plus', 3)
('des', 2)
('la', 6)
('Buonaparte.', 1)
('Non,', 2)
('vous', 10)


Spark существенно упростил реализацию многих задач, ранее решаемых с использованием MapReduce. Эта однострочная программа - WordCount - является наиболее популярным примером задачи, эффективно распараллеливаемой в Hadoop кластере. Её реализация в MapReduce занимается около 130 строк кода.

**Упражнение**. Улучшите процедуру, убирая из слов лишние символы и трансформируя все слова в нижний регистр. Используйте регулярные выражения.

In [17]:
import re

# Паттерн: латиница, кириллица (включая Ёё), цифры
word_pattern = re.compile(r'[a-zA-Zа-яА-ЯёЁ0-9]+')

wordCounts = linesWithWar \
    .flatMap(lambda line: word_pattern.findall(line.lower())) \
    .map(lambda word: (word, 1)) \
    .reduceByKey(lambda a, b: a + b)

wordCounts.saveAsTextFile("warandpeace_histogram2.txt")
!head warandpeace_histogram2.txt/part-00000

('genes', 1)
('et', 4)
('sont', 1)
('plus', 5)
('des', 2)
('поместья', 1)
('la', 7)
('non', 2)
('vous', 11)
('me', 1)


## Операции с множествами

Инициализируйте два множества

In [18]:
a = sc.parallelize([1,2,3,4])
b = sc.parallelize([3,4,6,7])

Найдите объединение a и b и соберите данные на главный узел с помощью функции collect

In [19]:
a.union(b).collect()

[1, 2, 3, 4, 3, 4, 6, 7]

Обратите внимание, что общие элементы дублируются, поэтому результат не является классическим множеством на самом деле. Такое поведение делает это операцию очень дешёвой, так как обновляется только информация о местонахождении данных для данного RDD. Уберите дубликаты с помощью `distinct`.

In [20]:
a.union(b).distinct().collect()

[4, 1, 2, 6, 3, 7]

Найдите пересечение множеств

In [21]:
a.intersection(b).collect()

[4, 3]

Найдите разность множеств

In [22]:
a.subtract(b).collect()

[1, 2]

*Примечание*. При запуске collect на центральный узел - driver передаются все данные из распределённого RDD. При работе с большим объемом данных выполнение данной команды может заполнить всю оперативную память driver узла.

**Упражнение**. Найдите в исходном коде Spark определение функции distinct. Объясните почему реализация этой операции действительно убирает дубликаты.

In [23]:
!grep -A 30 "def distinct" /content/spark-3.5.1-bin-hadoop3/python/pyspark/rdd.py

    def distinct(self: "RDD[T]", numPartitions: Optional[int] = None) -> "RDD[T]":
        """
        Return a new RDD containing the distinct elements in this RDD.

        .. versionadded:: 0.7.0

        Parameters
        ----------
        numPartitions : int, optional
            the number of partitions in new :class:`RDD`

        Returns
        -------
        :class:`RDD`
            a new :class:`RDD` containing the distinct elements

        See Also
        --------
        :meth:`RDD.countApproxDistinct`

        Examples
        --------
        >>> sorted(sc.parallelize([1, 1, 2, 3]).distinct().collect())
        [1, 2, 3]
        """
        return (
            self.map(lambda x: (x, None))
            .reduceByKey(lambda x, _: x, numPartitions)
            .map(lambda x: x[0])
        )



Реализация операции `distinct` гарантирует удаление дубликатов за счёт преобразования каждого элемента в ключ пары ключ-значение (`map`), после чего метод `reduceByKey` выполняет перемешивание данных (shuffle), группируя все идентичные ключи на одних и тех же узлах кластера. Функция редукции `lambda x, _: x` агрегирует значения для каждого ключа, игнорируя повторения и оставляя только одну запись на уникальный элемент, а финальный `map` извлекает эти уникальные ключи обратно в результирующий RDD.

## Общие переменные

В Apache Spark общими переменными являются широковещательные (broadcast) переменные и аккумулирующие (accumulator) переменные.

### Широковещательные переменные

Общие переменные удобны если вы обращаетесь к небольшому объёму данных на всех узлах. Например, это могут быть параметры алгоритмов обработки, небольшие матрицы.

В консоли, с которой вы работали в предыдущем разделе, создайте широковещательную переменную.

In [24]:
broadcastVar = sc.broadcast([1,2,3])

Для получения значения обратитесь к полю value:

In [25]:
broadcastVar.value

[1, 2, 3]

### Аккумулирующие переменные

Аккумулирующие переменные являются объектами, которые могут быть изменены только ассоциативной операцией добавления. Они используются для эффективной реализации счётчиков и суммарных значений. Вы можете также использовать свой тип, над котором определена ассоциативная операция при необходимости.

Особенностью использования переменной является возможность доступа к значению только на узле в driver процессе.

Потренируйтесь в создании аккумулирующих переменных:

In [26]:
accum = sc.accumulator(0)

Следующим шагом запустите параллельную обработку массива и в каждом параллельном задании добавьте к аккумулирующей переменной значение элемента массива:

In [27]:
sc.parallelize([1,2,3,4]).foreach(lambda x: accum.add(x))

Для получения текущего значения вызовите команду:

In [28]:
accum.value

10

Результатом должно быть число 10. Пары ключ-значение Создайте пару ключ-значение из двух букв:

In [29]:
pair = ('a', 'b')

Для доступа к первому значению обратитесь к полю 0:

In [30]:
pair[0]

'a'

Для доступа к второму значению к полю 1:

In [31]:
pair[1]

'b'

Если распределённая коллекция состоит из пар, то они трактуются как для ключ-значение и для таких коллекций доступны дополнительные операции. Наиболее распространённые, это: группировка по ключу, агрегирование значений с одинаковыми ключами, объединение двух коллекций по ключу.

## Топ-10 популярных номеров такси

Проанализируем данные о поездках такси в Нью-Йорке и найдём 10 номеров такси, которые совершили наибольшее количество поездок.

Загрузка данных nyctaxi.csv

In [32]:
!wget -O nyctaxi.csv https://git.ai.ssau.ru/tk/big_data/raw/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/nyctaxi.csv

--2026-03-12 18:47:13--  https://git.ai.ssau.ru/tk/big_data/raw/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/nyctaxi.csv
Resolving git.ai.ssau.ru (git.ai.ssau.ru)... 91.222.131.161
Connecting to git.ai.ssau.ru (git.ai.ssau.ru)|91.222.131.161|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: /tk/big_data/raw/commit/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/nyctaxi.csv [following]
--2026-03-12 18:47:14--  https://git.ai.ssau.ru/tk/big_data/raw/commit/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/nyctaxi.csv
Reusing existing connection to git.ai.ssau.ru:443.
HTTP request sent, awaiting response... 200 OK
Length: 79500408 (76M) [text/plain]
Saving to: ‘nyctaxi.csv’

nyctaxi.csv         100%[===================>]  75.82M   376KB/s    in 89s     

2026-03-12 18:48:44 (872 KB/s) - ‘nyctaxi.csv’ saved [79500408/79500408]



Создайте RDD на основе загруженных данных nyctaxi.csv:

In [33]:
taxi = sc.textFile("nyctaxi.csv")

Выведите первые 5 строк из данной таблицы:

In [34]:
for t in taxi.take(5):
    print(t)

"_id","_rev","dropoff_datetime","dropoff_latitude","dropoff_longitude","hack_license","medallion","passenger_count","pickup_datetime","pickup_latitude","pickup_longitude","rate_code","store_and_fwd_flag","trip_distance","trip_time_in_secs","vendor_id"
"29b3f4a30dea6688d4c289c9672cb996","1-ddfdec8050c7ef4dc694eeeda6c4625e","2013-01-11 22:03:00",+4.07033460000000E+001,-7.40144200000000E+001,"A93D1F7F8998FFB75EEF477EB6077516","68BC16A99E915E44ADA7E639B4DD5F59",2,"2013-01-11 21:48:00",+4.06760670000000E+001,-7.39810790000000E+001,1,,+4.08000000000000E+000,900,"VTS"
"2a80cfaa425dcec0861e02ae44354500","1-b72234b58a7b0018a1ec5d2ea0797e32","2013-01-11 04:28:00",+4.08190960000000E+001,-7.39467470000000E+001,"64CE1B03FDE343BB8DFB512123A525A4","60150AA39B2F654ED6F0C3AF8174A48A",1,"2013-01-11 04:07:00",+4.07280540000000E+001,-7.40020370000000E+001,1,,+8.53000000000000E+000,1260,"VTS"
"29b3f4a30dea6688d4c289c96758d87e","1-387ec30eac5abda89d2abefdf947b2c1","2013-01-11 22:02:00",+4.07277180000000E+00

Обратите внимание, что первая строка является заголовком. Её как правило нужно будет отфильтровать. Одним из эффективных способов является следующий:

In [35]:
import itertools
taxi.mapPartitionsWithIndex(lambda idx, it: itertools.islice(it,1,None) if (idx==0) else it)

PythonRDD[57] at RDD at PythonRDD.scala:53

*Примечание*. Для анализа структурированных табличных данных рассматривайте в качестве альтернативы использование SQL API и DataSet API.

Для разбора значений потребуется создать RDD, где каждая строка разбита на массив подстрок. Используйте запятую в качестве разделителя.

In [36]:
taxiParse = taxi.map(lambda line: line.split(","))

Теперь преобразуем массив строк в массив пар ключ-значение, где ключом будет служить номер такси (6 колонка), а значением единица.

In [37]:
taxiMedKey = taxiParse.map(lambda row: (row[6], 1))

Следом мы можем найти количество поездок каждого номера такси:

In [38]:
taxiMedCounts = taxiMedKey.reduceByKey(lambda v1, v2: v1+v2)

Выведем полученные результаты в отсортированном виде:

In [39]:
top10 = taxiMedCounts.map(lambda x: x[::-1]).top(10)
for x in top10:
    print(x[::-1])

('"AB44AD9A03B7CFAF3925103BDCC0AF23"', 44)
('"71CACFBADF9568AAE88A843DB511D172"', 41)
('"6483B9BFCB216EC88986EA3AB13064E7"', 41)
('"4C73459B430339981D78795300433438"', 41)
('"67E71D24AF704D814A0A825005ADA72E"', 40)
('"02E5A4136FD0A775A023A005A4EABC62"', 40)
('"9DFBCD218E7116F34C044F0680A0FB8A"', 39)
('"8DEB70907D00AA1D7FF5E2683240549B"', 39)
('"7989C2AB3F345F4AB54D3CF1E0480D67"', 39)
('"6C9F67DF658DC5636F9E7752F203F70A"', 39)


**Являются ли обе `map` операции распределёнными?**

Да, обе операции `map` являются распределёнными трансформациями: при их вызове функция сериализуется и отправляется на воркер-узлы, где применяется параллельно к элементам каждой партиции RDD. Благодаря ленивым вычислениям в Spark, эти трансформации не выполняются немедленно, а лишь формируют граф вычислений (DAG), который будет исполнен при вызове действия (например, `top()` или `count()`), что обеспечивает масштабирование обработки на весь кластер.

Найдите в документации Spark в классах RDD или PairRDDFunctions метод top.

In [40]:
!grep -A 7 "def top" /content/spark-3.5.1-bin-hadoop3/python/pyspark/rdd.py

    def top(self: "RDD[S]", num: int) -> List["S"]:
        ...

    @overload
    def top(self: "RDD[T]", num: int, key: Callable[[T], "S"]) -> List[T]:
        ...

    def top(self: "RDD[T]", num: int, key: Optional[Callable[[T], "S"]] = None) -> List[T]:
        """
        Get the top N elements from an RDD.

        .. versionadded:: 1.0.0

        Parameters
        ----------
--
        def topIterator(iterator: Iterable[T]) -> Iterable[List[T]]:
            yield heapq.nlargest(num, iterator, key=key)

        def merge(a: List[T], b: List[T]) -> List[T]:
            return heapq.nlargest(num, a + b, key=key)

        return self.mapPartitions(topIterator).reduce(merge)



Вы также можете сгруппировать все описанные выше трансформации, преобразующие исходные данные в одну цепочку:

In [41]:
taxiCounts = taxi.map(lambda line: line.split(",")).map(lambda row: (row[6],1)).reduceByKey(lambda a,b: a + b)

Попробуйте найти общее количество номеров такси несколько раз, предварительно объявив RDD taxiCounts как сохраняемую в кэше:

In [42]:
taxiCounts.cache()

PythonRDD[67] at RDD at PythonRDD.scala:53

Первый запуск (медленнее, т.к. вычисляется)

In [43]:
%%time
taxiCounts.count()

CPU times: user 6.65 ms, sys: 1.33 ms, total: 7.98 ms
Wall time: 3.25 s


13371

Второй запуск (быстрее, т.к. в кэше)

In [44]:
%%time
taxiCounts.count()

CPU times: user 6.59 ms, sys: 296 µs, total: 6.89 ms
Wall time: 777 ms


13371

## Настройка способа хранения RDD

В данной части будет рассмотрена настройка способов хранения RDD. Вы сравните различные способы хранения, включая: хранение в сериализованном виде, в исходном, с репликацией.

In [45]:
import pyspark
taxi.persist(storageLevel=pyspark.StorageLevel.MEMORY_ONLY)
print(taxi.getStorageLevel())

Memory Serialized 1x Replicated


Другими способами хранения являются:

- MEMORY_AND_DISK,
- MEMORY_AND_DISK_SER,
- DISK_ONLY,
- MEMORY_ONLY_2,
- MEMORY_AND_DISK_2,
- OFF_HEAP.

## Анализ данных велопарковок

1. Проведите анализ данных велопарковок на языке Python в интерактивном режиме из Jupyter книг:
- `L1_interactive_bike_analysis_python_with_rdd.ipynb`,
- `L1_interactive_bike_analysis_python_with_dataframes.ipynb`.

2. Проведите анализ данных велопарковок на языке Scala или Python в неинтерактивном режиме (`--deploy-mode cluster`). Инструкции по созданию и запуску приложений:
- `L1_noninteractive_bike_analysis_scala.py`

### Описание данных

https://www.kaggle.com/benhamner/sf-bay-area-bike-share

stations.csv схема:

```
id: station ID number
name: name of station
lat: latitude
long: longitude
dock_count: number of total docks at station
city: city (San Francisco, Redwood City, Palo Alto, Mountain View, San Jose)
installation_date: original date that station was installed. If station was moved, it is noted below.
```

trips.csv схема:

```
id: numeric ID of bike trip
duration: time of trip in seconds
start_date: start date of trip with date and time, in PST
start_station_name: station name of start station
start_station_id: numeric reference for start station
end_date: end date of trip with date and time, in PST
end_station_name: station name for end station
end_station_id: numeric reference for end station
bike_id: ID of bike used
subscription_type: Subscriber = annual or 30-day member; Customer = 24-hour or 3-day member
zip_code: Home zip code of subscriber (customers can choose to manually enter zip at kiosk however data is unreliable)
```

In [46]:
!wget -O stations.csv https://git.ai.ssau.ru/tk/big_data/raw/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/stations.csv

--2026-03-12 18:48:51--  https://git.ai.ssau.ru/tk/big_data/raw/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/stations.csv
Resolving git.ai.ssau.ru (git.ai.ssau.ru)... 91.222.131.161
Connecting to git.ai.ssau.ru (git.ai.ssau.ru)|91.222.131.161|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: /tk/big_data/raw/commit/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/stations.csv [following]
--2026-03-12 18:48:52--  https://git.ai.ssau.ru/tk/big_data/raw/commit/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/stations.csv
Reusing existing connection to git.ai.ssau.ru:443.
HTTP request sent, awaiting response... 200 OK
Length: 5647 (5.5K) [text/plain]
Saving to: ‘stations.csv’

stations.csv        100%[===================>]   5.51K  --.-KB/s    in 0s      

2026-03-12 18:48:52 (26.2 MB/s) - ‘stations.csv’ saved [5647/5647]



In [47]:
!wget -O trips.csv https://git.ai.ssau.ru/tk/big_data/raw/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/trips.csv

--2026-03-12 18:48:52--  https://git.ai.ssau.ru/tk/big_data/raw/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/trips.csv
Resolving git.ai.ssau.ru (git.ai.ssau.ru)... 91.222.131.161
Connecting to git.ai.ssau.ru (git.ai.ssau.ru)|91.222.131.161|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: /tk/big_data/raw/commit/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/trips.csv [following]
--2026-03-12 18:48:53--  https://git.ai.ssau.ru/tk/big_data/raw/commit/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/trips.csv
Reusing existing connection to git.ai.ssau.ru:443.
HTTP request sent, awaiting response... 200 OK
Length: 80208831 (76M) [text/plain]
Saving to: ‘trips.csv’

trips.csv           100%[===================>]  76.49M  1.47MB/s    in 44s     

2026-03-12 18:49:38 (1.74 MB/s) - ‘trips.csv’ saved [80208831/80208831]



In [48]:
tripData = sc.textFile("trips.csv")
# запомним заголовок, чтобы затем его исключить из данных
tripsHeader = tripData.first()
trips = tripData.filter(lambda row: row != tripsHeader).map(lambda row: row.split(",", -1))

stationData = sc.textFile("stations.csv")
stationsHeader = stationData.first()
stations = stationData.filter(lambda row: row != stationsHeader).map(lambda row: row.split(",", -1))

In [49]:
list(enumerate(tripsHeader.split(",")))

[(0, 'id'),
 (1, 'duration'),
 (2, 'start_date'),
 (3, 'start_station_name'),
 (4, 'start_station_id'),
 (5, 'end_date'),
 (6, 'end_station_name'),
 (7, 'end_station_id'),
 (8, 'bike_id'),
 (9, 'subscription_type'),
 (10, 'zip_code')]

In [50]:
list(enumerate(stationsHeader.split(",")))

[(0, 'id'),
 (1, 'name'),
 (2, 'lat'),
 (3, 'long'),
 (4, 'dock_count'),
 (5, 'city'),
 (6, 'installation_date')]

In [51]:
trips.take(2)

[['4576',
  '63',
  '',
  'South Van Ness at Market',
  '66',
  '8/29/2013 14:14',
  'South Van Ness at Market',
  '66',
  '520',
  'Subscriber',
  '94127'],
 ['4607',
  '',
  '8/29/2013 14:42',
  'San Jose City Hall',
  '10',
  '8/29/2013 14:43',
  'San Jose City Hall',
  '10',
  '661',
  'Subscriber',
  '95138']]

In [52]:
stations.take(2)

[['2',
  'San Jose Diridon Caltrain Station',
  '37.329732',
  '-121.90178200000001',
  '27',
  'San Jose',
  '8/6/2013'],
 ['3',
  'San Jose Civic Center',
  '37.330698',
  '-121.888979',
  '15',
  'San Jose',
  '8/5/2013']]

Объявите `stationsIndexed` так, чтобы результатом был список пар ключ-значение с целочисленным ключом  из первой колонки.  Таким образом вы создаёте индекс на основе первой колонки - номера велостоянки

In [53]:
stationsIndexed = stations.keyBy(lambda station: station[0])

In [54]:
stationsIndexed.take(2)

[('2',
  ['2',
   'San Jose Diridon Caltrain Station',
   '37.329732',
   '-121.90178200000001',
   '27',
   'San Jose',
   '8/6/2013']),
 ('3',
  ['3',
   'San Jose Civic Center',
   '37.330698',
   '-121.888979',
   '15',
   'San Jose',
   '8/5/2013'])]

Аналогичное действие проделайте для индексирования коллекции trips по колонкам start_station_id и  end_station_id и сохраните результат в переменные, например tripsByStartTerminals и tripsByEndTerminals.

In [55]:
tripsByStartTerminals = trips.keyBy(lambda trip: trip[4])
tripsByEndTerminals = trips.keyBy(lambda trip: trip[7])

Выполните операцию объединения коллекций по ключу с помощью функции join. Объедините stationsIndexed и tripsByStartTerminals, stationsIndexed и tripsByEndTerminals.

In [56]:
startTrips = stationsIndexed.join(tripsByStartTerminals)
endTrips = stationsIndexed.join(tripsByEndTerminals)

Объявление последовательности трансформаций приводит к созданию ацикличного ориентированного графа. Вывести  полученный граф можно для любого RDD.

In [57]:
print(startTrips.toDebugString().decode("utf-8"))

(5) PythonRDD[93] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[85] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[84] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(5) PairwiseRDD[83] at join at /tmp/ipykernel_89151/210173577.py:1 []
    |  PythonRDD[82] at join at /tmp/ipykernel_89151/210173577.py:1 []
    |  UnionRDD[81] at union at NativeMethodAccessorImpl.java:0 []
    |  PythonRDD[79] at RDD at PythonRDD.scala:53 []
    |  stations.csv MapPartitionsRDD[74] at textFile at NativeMethodAccessorImpl.java:0 []
    |  stations.csv HadoopRDD[73] at textFile at NativeMethodAccessorImpl.java:0 []
    |  PythonRDD[80] at RDD at PythonRDD.scala:53 []
    |  trips.csv MapPartitionsRDD[71] at textFile at NativeMethodAccessorImpl.java:0 []
    |  trips.csv HadoopRDD[70] at textFile at NativeMethodAccessorImpl.java:0 []


In [58]:
print(endTrips.toDebugString().decode("utf-8"))

(5) PythonRDD[94] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[92] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[91] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(5) PairwiseRDD[90] at join at /tmp/ipykernel_89151/210173577.py:2 []
    |  PythonRDD[89] at join at /tmp/ipykernel_89151/210173577.py:2 []
    |  UnionRDD[88] at union at NativeMethodAccessorImpl.java:0 []
    |  PythonRDD[86] at RDD at PythonRDD.scala:53 []
    |  stations.csv MapPartitionsRDD[74] at textFile at NativeMethodAccessorImpl.java:0 []
    |  stations.csv HadoopRDD[73] at textFile at NativeMethodAccessorImpl.java:0 []
    |  PythonRDD[87] at RDD at PythonRDD.scala:53 []
    |  trips.csv MapPartitionsRDD[71] at textFile at NativeMethodAccessorImpl.java:0 []
    |  trips.csv HadoopRDD[70] at textFile at NativeMethodAccessorImpl.java:0 []


Выполните  объявленные графы трансформаций вызовом команды count.

In [59]:
startTrips.count()

669959

In [60]:
endTrips.count()

669959

Если вы знаете распределение ключей заранее, вы можете выбрать оптимальный способ хеширования ключей по разделам `Partition`. Например, если один ключ встречается на порядки чаще, чем другие ключи, то использование `HashPartitioner` будет не лучшим выбором, так как данные связанные с этим ключом будут собираться в одном разделе. Это приведёт к неравномерной нагрузке на вычислительные ресурсы.

Выбрать определённую реализацию класса распределения по разделам можно с помощью функции RDD `partitionBy`. Например, для RDD `stationsIndexed`  выбирается `portable_hash(idx)` с количеством разделов равным количеству разделов trips RDD.

In [61]:
from pyspark.rdd import portable_hash

stationsIndexed.partitionBy(numPartitions=trips.getNumPartitions(), partitionFunc=lambda x: portable_hash(x[0]))

MapPartitionsRDD[100] at mapPartitions at PythonRDD.scala:160

Узнать какой класс назначен для текущего RDD можно обращением к полю partitioner.

In [62]:
stationsIndexed.partitioner

### Создание модели данных

Для более эффективной  обработки и получения дополнительных возможностей мы можем объявить классы сущностей предметной области и преобразовать исходные строковые данные в объявленное представление.

In [63]:
from typing import NamedTuple
from datetime import datetime

In [64]:
def initStation(stations):
    class Station(NamedTuple):
        station_id: int
        name: str
        lat: float
        long: float
        dockcount: int
        landmark: str
        installation: str

    for station in stations:
        yield Station(
            station_id = int(station[0]),
            name = station[1],
            lat = float(station[2]),
            long = float(station[3]),
            dockcount = int(station[4]),
            landmark = station[5],
            installation = datetime.strptime(station[6], '%m/%d/%Y')
        )

In [65]:
stationsInternal = stations.mapPartitions(initStation)
stationsInternal.first()

Station(station_id=2, name='San Jose Diridon Caltrain Station', lat=37.329732, long=-121.90178200000001, dockcount=27, landmark='San Jose', installation=datetime.datetime(2013, 8, 6, 0, 0))

In [66]:
def initTrip(trips):
    class Trip(NamedTuple):
        trip_id: int
        duration: int
        start_date: datetime
        start_station_name: str
        start_station_id: int
        end_date: datetime
        end_station_name: str
        end_station_id: int
        bike_id: int
        subscription_type: str
        zip_code: str

    for trip in trips:
        try:
            yield Trip(
             trip_id = int(trip[0]),
             duration = int(trip[1]),
             start_date = datetime.strptime(trip[2], '%m/%d/%Y %H:%M'),
             start_station_name = trip[3],
             start_station_id = int(trip[4]),
             end_date = datetime.strptime(trip[5], '%m/%d/%Y %H:%M'),
             end_station_name = trip[6],
             end_station_id = trip[7],
             bike_id = int(trip[8]),
             subscription_type = trip[9],
             zip_code = trip[10]
            )
        except:
            pass

In [67]:
tripsInternal = trips.mapPartitions(initTrip)
tripsInternal.take(10)

[Trip(trip_id=4130, duration=71, start_date=datetime.datetime(2013, 8, 29, 10, 16), start_station_name='Mountain View City Hall', start_station_id=27, end_date=datetime.datetime(2013, 8, 29, 10, 17), end_station_name='Mountain View City Hall', end_station_id='27', bike_id=48, subscription_type='Subscriber', zip_code='97214'),
 Trip(trip_id=4251, duration=77, start_date=datetime.datetime(2013, 8, 29, 11, 29), start_station_name='San Jose City Hall', start_station_id=10, end_date=datetime.datetime(2013, 8, 29, 11, 30), end_station_name='San Jose City Hall', end_station_id='10', bike_id=26, subscription_type='Subscriber', zip_code='95060'),
 Trip(trip_id=4299, duration=83, start_date=datetime.datetime(2013, 8, 29, 12, 2), start_station_name='South Van Ness at Market', start_station_id=66, end_date=datetime.datetime(2013, 8, 29, 12, 4), end_station_name='Market at 10th', end_station_id='67', bike_id=319, subscription_type='Subscriber', zip_code='94103'),
 Trip(trip_id=4927, duration=103, s

Для каждой стартовой станции найдем среднее время поездки. Будем использовать метод groupByKey.

Для этого потребуется преобразовать trips RDD в RDD коллекцию пар ключ-значение аналогично тому, как мы совершали это ранее методом keyBy.

In [68]:
tripsByStartStation = tripsInternal.keyBy(lambda trip: trip.start_station_name)

Рассчитаем среднее время поездки для каждого стартового парковочного места

In [69]:
import numpy as np

avgDurationByStartStation = tripsByStartStation\
 .mapValues(lambda trip: trip.duration)\
 .groupByKey()\
 .mapValues(lambda trip_durations: np.mean(list(trip_durations)))

Выведем первые 10 результатов

In [70]:
%%time
avgDurationByStartStation.top(10, key=lambda x: x[1])

CPU times: user 10.3 ms, sys: 4.05 ms, total: 14.3 ms
Wall time: 16.5 s


[('University and Emerson', np.float64(7090.239417989418)),
 ('California Ave Caltrain Station', np.float64(4628.005847953216)),
 ('Redwood City Public Library', np.float64(4579.234741784037)),
 ('Park at Olive', np.float64(4438.1613333333335)),
 ('San Jose Civic Center', np.float64(4208.016938519448)),
 ('Rengstorff Avenue / California Street', np.float64(4174.082373782108)),
 ('Redwood City Medical Center', np.float64(3959.491961414791)),
 ('Palo Alto Caltrain Station', np.float64(3210.6489815253435)),
 ('San Mateo County Center', np.float64(2716.7700348432054)),
 ('Broadway at Main', np.float64(2481.2537313432836))]

Выполнение операции groupByKey приводит к интенсивным передачам данных. Если группировка делается для последующей редукции элементов лучше использовать трансформацию reduceByKey или aggregateByKey. Их выполнение приведёт сначала к локальной редукции над разделом Partition, а затем будет произведено окончательное суммирование над полученными частичными суммами.

*Примечание.* Выполнение reduceByKey логически сходно с выполнением Combine и Reduce фазы MapReduce  работы.

Функция aggregateByKey является аналогом reduceByKey с возможностью указывать начальный элемент.

Рассчитаем среднее значение с помощью aggregateByKey. Одновременно будут вычисляться два значения для каждого стартового терминала: сумма времён и количество поездок.

In [71]:
? tripsByStartStation.aggregateByKey

In [72]:
def seqFunc(acc, duration):
    duration_sum, count = acc
    return (duration_sum + duration, count + 1)

def combFunc(acc1, acc2):
    duration_sum1, count1 = acc1
    duration_sum2, count2 = acc2
    return (duration_sum1+duration_sum2, count1+count2)

def meanFunc(acc):
    duration_sum, count = acc
    return duration_sum/count

avgDurationByStartStation2 = tripsByStartStation\
  .mapValues(lambda trip: trip.duration)\
  .aggregateByKey(
    zeroValue=(0,0),
    seqFunc=seqFunc,
    combFunc=combFunc)\
  .mapValues(meanFunc)

В `zeroValue` передаётся начальное значение. В нашем случае это пара нулей. Первая функция `seqFunc` предназначена для прохода по коллекции партиции. На этом проходе значение элементов помещаются средой в переменную duration, а переменная «аккумулятора» acc накапливает значения. Вторая функция `combFunc` предназначена для этапа редукции частично посчитанных локальных результатов.

Сравните результаты `avgDurationByStartStation` и `avgDurationByStartStation2` и их время выполнения.

In [73]:
%%time
avgDurationByStartStation2.top(10, key=lambda x: x[1])

CPU times: user 7.88 ms, sys: 4.22 ms, total: 12.1 ms
Wall time: 16.7 s


[('University and Emerson', 7090.239417989418),
 ('California Ave Caltrain Station', 4628.005847953216),
 ('Redwood City Public Library', 4579.234741784037),
 ('Park at Olive', 4438.1613333333335),
 ('San Jose Civic Center', 4208.016938519448),
 ('Rengstorff Avenue / California Street', 4174.082373782108),
 ('Redwood City Medical Center', 3959.491961414791),
 ('Palo Alto Caltrain Station', 3210.6489815253435),
 ('San Mateo County Center', 2716.7700348432054),
 ('Broadway at Main', 2481.2537313432836)]

Теперь найдём первую поездку для каждой велостоянки. Для решения опять потребуется группировка. Ещё одним недостатком `groupByKey` данных является то, что для группировки данные должны поместиться в оперативной памяти. Это может привести к ошибке `OutOfMemoryException` для больших объёмов данных.

Найдем самую раннюю поездку для каждой станции. Сгруппируем поездки по станциям, возьмём первую поездку из отсортированного списка:

In [74]:
def earliestTrip(trips):
    if trips is None:
        return None
    if len(trips)==0:
        return trips
    trips = list(trips)
    min_date = trips[0].start_date
    min_trip = trips[0]
    for trip in trips[1:]:
        if min_date > trip.start_date:
            min_date = trip.start_date
            min_trip = trip
    return min_trip

firstGrouped = tripsByStartStation\
  .groupByKey()\
  .mapValues(lambda trips: earliestTrip(trips))

In [75]:
%%time
firstGrouped.take(5)

CPU times: user 11 ms, sys: 2.67 ms, total: 13.6 ms
Wall time: 26.8 s


[('Mountain View City Hall',
  Trip(trip_id=4081, duration=218, start_date=datetime.datetime(2013, 8, 29, 9, 38), start_station_name='Mountain View City Hall', start_station_id=27, end_date=datetime.datetime(2013, 8, 29, 9, 41), end_station_name='Mountain View City Hall', end_station_id='27', bike_id=150, subscription_type='Subscriber', zip_code='97214')),
 ('Santa Clara at Almaden',
  Trip(trip_id=4500, duration=109, start_date=datetime.datetime(2013, 8, 29, 13, 25), start_station_name='Santa Clara at Almaden', start_station_id=4, end_date=datetime.datetime(2013, 8, 29, 13, 27), end_station_name='Adobe on Almaden', end_station_id='5', bike_id=679, subscription_type='Subscriber', zip_code='95112')),
 ('San Salvador at 1st',
  Trip(trip_id=4517, duration=379, start_date=datetime.datetime(2013, 8, 29, 13, 34), start_station_name='San Salvador at 1st', start_station_id=8, end_date=datetime.datetime(2013, 8, 29, 13, 41), end_station_name='San Jose City Hall', end_station_id='10', bike_id=6

Лучшим вариантом с точки зрения эффективности будет использование трансформации `reduceByKey`

In [76]:
firstGrouped = tripsByStartStation\
  .reduceByKey(lambda tripA, tripB: tripA if tripA.start_date < tripB.start_date else tripB)

In [77]:
%%time
firstGrouped.take(5)

CPU times: user 8.28 ms, sys: 1.73 ms, total: 10 ms
Wall time: 16.5 s


[('Mountain View City Hall',
  Trip(trip_id=4081, duration=218, start_date=datetime.datetime(2013, 8, 29, 9, 38), start_station_name='Mountain View City Hall', start_station_id=27, end_date=datetime.datetime(2013, 8, 29, 9, 41), end_station_name='Mountain View City Hall', end_station_id='27', bike_id=150, subscription_type='Subscriber', zip_code='97214')),
 ('Santa Clara at Almaden',
  Trip(trip_id=4500, duration=109, start_date=datetime.datetime(2013, 8, 29, 13, 25), start_station_name='Santa Clara at Almaden', start_station_id=4, end_date=datetime.datetime(2013, 8, 29, 13, 27), end_station_name='Adobe on Almaden', end_station_id='5', bike_id=679, subscription_type='Subscriber', zip_code='95112')),
 ('San Salvador at 1st',
  Trip(trip_id=4517, duration=379, start_date=datetime.datetime(2013, 8, 29, 13, 34), start_station_name='San Salvador at 1st', start_station_id=8, end_date=datetime.datetime(2013, 8, 29, 13, 41), end_station_name='San Jose City Hall', end_station_id='10', bike_id=6

### Пример чтения csv файлов и работы с дефектными данными

Список опций чтения и записи для CSV файлов https://spark.apache.org/docs/latest/sql-data-sources-csv.html#data-source-option

Формат паттерна временной метки Spark SQL отличается от python библиотеки datetime.
https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html

In [78]:
tripData = spark.read\
.option("header", True)\
.option("inferSchema", True)\
.option("timestampFormat", 'M/d/y H:m')\
.csv("trips.csv")

tripData

DataFrame[id: int, duration: int, start_date: timestamp, start_station_name: string, start_station_id: int, end_date: timestamp, end_station_name: string, end_station_id: int, bike_id: int, subscription_type: string, zip_code: string]

In [79]:
tripData.printSchema()

root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)



In [80]:
tripData.show(n=5)

+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|  id|duration|         start_date|  start_station_name|start_station_id|           end_date|    end_station_name|end_station_id|bike_id|subscription_type|zip_code|
+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|4576|      63|               NULL|South Van Ness at...|              66|2013-08-29 14:14:00|South Van Ness at...|            66|    520|       Subscriber|   94127|
|4607|    NULL|2013-08-29 14:42:00|  San Jose City Hall|              10|2013-08-29 14:43:00|  San Jose City Hall|            10|    661|       Subscriber|   95138|
|4130|      71|2013-08-29 10:16:00|Mountain View Cit...|              27|2013-08-29 10:17:00|Mountain View Cit...|            27|     48|       Subscriber|   97214|
|4251|    

In [81]:
? tripData.dropna

In [82]:
tripData.dropna().show(n=5)

+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|  id|duration|         start_date|  start_station_name|start_station_id|           end_date|    end_station_name|end_station_id|bike_id|subscription_type|zip_code|
+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|4130|      71|2013-08-29 10:16:00|Mountain View Cit...|              27|2013-08-29 10:17:00|Mountain View Cit...|            27|     48|       Subscriber|   97214|
|4251|      77|2013-08-29 11:29:00|  San Jose City Hall|              10|2013-08-29 11:30:00|  San Jose City Hall|            10|     26|       Subscriber|   95060|
|4299|      83|2013-08-29 12:02:00|South Van Ness at...|              66|2013-08-29 12:04:00|      Market at 10th|            67|    319|       Subscriber|   94103|
|4927|    

In [83]:
tripData.describe().show()

+-------+-----------------+------------------+--------------------+------------------+--------------------+------------------+------------------+-----------------+--------------------+
|summary|               id|          duration|  start_station_name|  start_station_id|    end_station_name|    end_station_id|           bike_id|subscription_type|            zip_code|
+-------+-----------------+------------------+--------------------+------------------+--------------------+------------------+------------------+-----------------+--------------------+
|  count|           669959|            669958|              669959|            669959|              669959|            669959|            669959|           669959|              663340|
|   mean|460382.0098991132| 1107.951395460611|                NULL| 57.85187601032302|                NULL|57.837437813358726|427.58761954089726|             NULL|  1576245.3147059095|
| stddev|264584.4584872604|22255.453593552545|                NULL|17.11247

In [84]:
stationData = spark.read\
.option("header", True)\
.option("inferSchema", True)\
.option("timestampFormat", 'M/d/y')\
.csv("stations.csv")

stationData.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: timestamp (nullable = true)



In [85]:
stationData.show(n=5)

+---+--------------------+------------------+-------------------+----------+--------+-------------------+
| id|                name|               lat|               long|dock_count|    city|  installation_date|
+---+--------------------+------------------+-------------------+----------+--------+-------------------+
|  2|San Jose Diridon ...|         37.329732|-121.90178200000001|        27|San Jose|2013-08-06 00:00:00|
|  3|San Jose Civic Ce...|         37.330698|        -121.888979|        15|San Jose|2013-08-05 00:00:00|
|  4|Santa Clara at Al...|         37.333988|        -121.894902|        11|San Jose|2013-08-06 00:00:00|
|  5|    Adobe on Almaden|         37.331415|          -121.8932|        19|San Jose|2013-08-05 00:00:00|
|  6|    San Pedro Square|37.336721000000004|        -121.894074|        15|San Jose|2013-08-07 00:00:00|
+---+--------------------+------------------+-------------------+----------+--------+-------------------+
only showing top 5 rows



In [86]:
stationData.describe().show()

+-------+------------------+--------------------+-------------------+-------------------+-----------------+-------------+
|summary|                id|                name|                lat|               long|       dock_count|         city|
+-------+------------------+--------------------+-------------------+-------------------+-----------------+-------------+
|  count|                70|                  70|                 70|                 70|               70|           70|
|   mean|              43.0|                NULL|  37.59024338428572|-122.21841616428571|17.65714285714286|         NULL|
| stddev|24.166091947189145|                NULL|0.20347253639672502|0.20944604979644524|4.010441857493954|         NULL|
|    min|                 2|       2nd at Folsom|          37.329732|        -122.418954|               11|Mountain View|
|    max|                84|Yerba Buena Cente...|           37.80477|        -121.877349|               27|     San Jose|
+-------+---------------

### Пример использования DataFrame API

Выполните операцию объединения коллекций по ключу с помощью функции join. Объедините stationsIndexed и tripsByStartTerminals, stationsIndexed и tripsByEndTerminals.

https://spark.apache.org/docs/latest/sql-getting-started.html#untyped-dataset-operations-aka-dataframe-operations

In [87]:
tripData.printSchema()
stationData.printSchema()

root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: timestamp (nullable = true)



In [88]:
stationsView = stationData.select(stationData['id'], stationData['name'], stationData['lat'], stationData['long'])
stationsView.show()

+---+--------------------+------------------+-------------------+
| id|                name|               lat|               long|
+---+--------------------+------------------+-------------------+
|  2|San Jose Diridon ...|         37.329732|-121.90178200000001|
|  3|San Jose Civic Ce...|         37.330698|        -121.888979|
|  4|Santa Clara at Al...|         37.333988|        -121.894902|
|  5|    Adobe on Almaden|         37.331415|          -121.8932|
|  6|    San Pedro Square|37.336721000000004|        -121.894074|
|  7|Paseo de San Antonio|         37.333798|-121.88694299999999|
|  8| San Salvador at 1st|         37.330165|-121.88583100000001|
|  9|           Japantown|         37.348742|-121.89471499999999|
| 10|  San Jose City Hall|         37.337391|        -121.886995|
| 11|         MLK Library|         37.335885|-121.88566000000002|
| 12|SJSU 4th at San C...|         37.332808|-121.88389099999999|
| 13|       St James Park|         37.339301|-121.88993700000002|
| 14|Arena

In [89]:
startTrips = tripData.select(
    tripData.id,
    tripData.duration,
    tripData.start_station_id
).withColumnRenamed('id', 'trip_id').join(stationsView, tripData.start_station_id == stationsView.id)
startTrips = startTrips.drop('id')

In [90]:
startTrips.show()

+-------+--------+----------------+--------------------+------------------+-------------------+
|trip_id|duration|start_station_id|                name|               lat|               long|
+-------+--------+----------------+--------------------+------------------+-------------------+
|   4576|      63|              66|South Van Ness at...|         37.774814|        -122.418954|
|   4607|    NULL|              10|  San Jose City Hall|         37.337391|        -121.886995|
|   4130|      71|              27|Mountain View Cit...|         37.389218|        -122.081896|
|   4251|      77|              10|  San Jose City Hall|         37.337391|        -121.886995|
|   4299|      83|              66|South Van Ness at...|         37.774814|        -122.418954|
|   4927|     103|              59| Golden Gate at Polk|         37.781332|        -122.418603|
|   4500|     109|               4|Santa Clara at Al...|         37.333988|        -121.894902|
|   4563|     111|               8| San 

Объединение trips с stations по конечной станции

In [91]:
endTrips = tripData.select(
    tripData.id,
    tripData.duration,
    tripData.end_station_id
).withColumnRenamed('id', 'trip_id').join(stationsView, tripData.end_station_id == stationsView.id)
endTrips = endTrips.drop('id')

In [92]:
endTrips.show()

+-------+--------+--------------+--------------------+------------------+-------------------+
|trip_id|duration|end_station_id|                name|               lat|               long|
+-------+--------+--------------+--------------------+------------------+-------------------+
|   4576|      63|            66|South Van Ness at...|         37.774814|        -122.418954|
|   4607|    NULL|            10|  San Jose City Hall|         37.337391|        -121.886995|
|   4130|      71|            27|Mountain View Cit...|         37.389218|        -122.081896|
|   4251|      77|            10|  San Jose City Hall|         37.337391|        -121.886995|
|   4299|      83|            67|      Market at 10th|37.776619000000004|-122.41738500000001|
|   4927|     103|            59| Golden Gate at Polk|         37.781332|        -122.418603|
|   4500|     109|             5|    Adobe on Almaden|         37.331415|          -121.8932|
|   4563|     111|             8| San Salvador at 1st|      

### Пример использования Spark SQL API

In [93]:
stationData.createOrReplaceTempView("stations")
tripData.createOrReplaceTempView("trips")

In [94]:
endTrips = spark.sql("""
SELECT trips.id as trip_id, trips.end_station_id, trips.duration, stations.name as station_name, stations.lat, stations.long
FROM trips INNER JOIN stations
    ON trips.end_station_id==stations.id
""")

In [95]:
endTrips.show()

+-------+--------------+--------+--------------------+------------------+-------------------+
|trip_id|end_station_id|duration|        station_name|               lat|               long|
+-------+--------------+--------+--------------------+------------------+-------------------+
|   4576|            66|      63|South Van Ness at...|         37.774814|        -122.418954|
|   4607|            10|    NULL|  San Jose City Hall|         37.337391|        -121.886995|
|   4130|            27|      71|Mountain View Cit...|         37.389218|        -122.081896|
|   4251|            10|      77|  San Jose City Hall|         37.337391|        -121.886995|
|   4299|            67|      83|      Market at 10th|37.776619000000004|-122.41738500000001|
|   4927|            59|     103| Golden Gate at Polk|         37.781332|        -122.418603|
|   4500|             5|     109|    Adobe on Almaden|         37.331415|          -121.8932|
|   4563|             8|     111| San Salvador at 1st|      

Для каждой стартовой станции найдем среднее время поездки.

Рассчитаем среднее время поездки для каждого стартового парковочного места

In [96]:
spark.sql("""
SELECT start_station_name, avg(duration)
FROM trips
GROUP BY trips.start_station_name
ORDER BY avg(duration) DESC
""").show()

+--------------------+------------------+
|  start_station_name|     avg(duration)|
+--------------------+------------------+
|University and Em...| 7090.239417989418|
|California Ave Ca...| 4628.005847953216|
|Redwood City Publ...| 4579.234741784037|
|       Park at Olive|4438.1613333333335|
|San Jose Civic Ce...| 4208.016938519448|
|Rengstorff Avenue...| 4174.082373782108|
|Redwood City Medi...| 3959.491961414791|
|Palo Alto Caltrai...|3210.6489815253435|
|San Mateo County ...|2716.7700348432054|
|    Broadway at Main|2481.2537313432836|
|Cowper at University|2477.2379912663755|
|Redwood City Calt...|2415.7175032175032|
|South Van Ness at...| 2401.603338509317|
|San Antonio Caltr...| 2377.497487437186|
|San Antonio Shopp...| 2285.981298129813|
|           Japantown| 2207.809947643979|
|Washington at Kea...|1979.3077445652175|
|Arena Green / SAP...|1963.9679144385027|
|SJSU 4th at San C...| 1962.280341880342|
|Castro Street and...|1868.3135135135135|
+--------------------+------------

### Пример подготовки данных c Spark SQL, pandas, h3 для их визуализации на карте folium

In [97]:
!pip install h3 h3_pyspark pandas folium

Найдём велосипеды, которые ездили в рождество 2014 года.
https://spark.apache.org/docs/latest/api/sql/#make_timestamp

In [98]:
# year - the year to represent, from 1 to 9999
# month - the month-of-year to represent, from 1 (January) to 12 (December)
# day - the day-of-month to represent, from 1 to 31
# hour - the hour-of-day to represent, from 0 to 23
# min - the minute-of-hour to represent, from 0 to 59
# sec - the second-of-minute and its micro-fraction to represent, from 0 to 60. The value can be either an integer like 13 , or a fraction like 13.123. If the sec argument equals to 60, the seconds field is set to 0 and 1 minute is added to the final timestamp.
# timezone - the time zone identifier. For example, CET, UTC and etc.

spark.sql("""
SELECT bike_id, start_date, end_date
FROM trips
WHERE
    start_date > make_timestamp(2014, 12, 25, 0, 0, 0)
    AND start_date <  make_timestamp(2014, 12, 26, 0, 0, 0)
""").show()

+-------+-------------------+-------------------+
|bike_id|         start_date|           end_date|
+-------+-------------------+-------------------+
|    379|2014-12-25 22:10:00|2014-12-25 22:18:00|
|    709|2014-12-25 21:21:00|2014-12-25 21:27:00|
|    376|2014-12-25 20:40:00|2014-12-25 20:46:00|
|    541|2014-12-25 20:27:00|2014-12-25 20:32:00|
|    283|2014-12-25 19:56:00|2014-12-25 20:01:00|
|    519|2014-12-25 19:56:00|2014-12-25 20:01:00|
|    583|2014-12-25 19:05:00|2014-12-25 19:07:00|
|    495|2014-12-25 18:42:00|2014-12-25 18:44:00|
|    541|2014-12-25 18:28:00|2014-12-25 18:37:00|
|    585|2014-12-25 18:27:00|2014-12-25 18:37:00|
|    574|2014-12-25 18:12:00|2014-12-25 18:21:00|
|    630|2014-12-25 18:12:00|2014-12-25 18:22:00|
|    583|2014-12-25 18:05:00|2014-12-25 18:22:00|
|    290|2014-12-25 18:01:00|2014-12-25 18:15:00|
|    451|2014-12-25 17:55:00|2014-12-25 18:04:00|
|    630|2014-12-25 17:55:00|2014-12-25 17:59:00|
|    574|2014-12-25 17:54:00|2014-12-25 17:59:00|


Найдём станции через которые проехал один из велосипедов, найденных ранее.

In [99]:
spark.sql("""
SELECT trips.bike_id, trips.start_date, trips.end_date, stations.name
FROM trips INNER JOIN stations
    ON trips.start_station_id == stations.id
WHERE
    bike_id == 583
    AND start_date > make_timestamp(2014, 12, 25, 0, 0, 0)
    AND start_date <  make_timestamp(2014, 12, 26, 0, 0, 0)
""").show()

+-------+-------------------+-------------------+--------------+
|bike_id|         start_date|           end_date|          name|
+-------+-------------------+-------------------+--------------+
|    583|2014-12-25 19:05:00|2014-12-25 19:07:00|Market at 10th|
|    583|2014-12-25 18:05:00|2014-12-25 18:22:00|Market at 10th|
|    583|2014-12-25 12:14:00|2014-12-25 12:21:00| Market at 4th|
+-------+-------------------+-------------------+--------------+



Найдём все станции, которые попали в ту же клетку h3 координатной сетки что и станции, через которые проехал велосипед 583 25.12.2014.

Отобразим gps координаты станций в координаты h3.

In [100]:
from pyspark.sql import functions as F
import h3_pyspark
import h3

H3 Grid Resolutions https://h3geo.org/docs/core-library/restable/

In [101]:
h3_udf = F.udf(lambda lat, lng: h3.latlng_to_cell(lat, lng, 8), F.StringType())

stationData.withColumn('h3', h3_udf(F.col('lat'), F.col('long'))) \
    .createOrReplaceTempView("stations_h3")

Используя вложенный sql запрос, найдём h3 координаты станций, через который проехал велосипед 583. А затем отфильтруем поездки в рождество 2014 года, которые стартовали со станций с теми же h3 координатами, что мы нашли.

In [102]:
christmas_583_contacts = spark.sql("""
SELECT trips.bike_id, trips.start_date, stations_h3.h3, stations_h3.lat, stations_h3.long, stations_h3.name
FROM trips INNER JOIN stations_h3
    ON trips.start_station_id == stations_h3.id
WHERE
    stations_h3.h3 in (SELECT stations_h3.h3
                            FROM trips INNER JOIN stations_h3
                                ON trips.start_station_id == stations_h3.id
                            WHERE
                                bike_id == 583
                                AND start_date > make_timestamp(2014, 12, 25, 0, 0, 0)
                                AND start_date <  make_timestamp(2014, 12, 26, 0, 0, 0))
    AND start_date > make_timestamp(2014, 12, 25, 0, 0, 0)
    AND start_date < make_timestamp(2014, 12, 26, 0, 0, 0)
ORDER BY trips.start_date
""")
christmas_583_contacts.cache()
christmas_583_contacts.show()

+-------+-------------------+---------------+------------------+-------------------+------------------+
|bike_id|         start_date|             h3|               lat|               long|              name|
+-------+-------------------+---------------+------------------+-------------------+------------------+
|    439|2014-12-25 01:40:00|8828308281fffff|37.776619000000004|-122.41738500000001|    Market at 10th|
|    659|2014-12-25 09:49:00|88283082abfffff|37.781752000000004|-122.40512700000001|     5th at Howard|
|    465|2014-12-25 09:49:00|88283082abfffff|37.781752000000004|-122.40512700000001|     5th at Howard|
|    583|2014-12-25 12:14:00|88283082abfffff|         37.786305|-122.40496599999999|     Market at 4th|
|    479|2014-12-25 12:22:00|88283082abfffff|37.781752000000004|-122.40512700000001|     5th at Howard|
|    331|2014-12-25 12:22:00|88283082abfffff|37.781752000000004|-122.40512700000001|     5th at Howard|
|    330|2014-12-25 12:27:00|88283082abfffff|37.781752000000004|

In [103]:
import pandas as pd

h3_places = christmas_583_contacts.select('lat','long', 'name', 'h3').toPandas()

In [104]:
import folium

def init_map(hexagons, width=1100, height=900):
    lats = []
    longs = []
    for hexagon in hexagons:
        if hexagon:
            lat, lng = h3.cell_to_latlng(hexagon)
            lats.append(lat)
            longs.append(lng)

    return folium.Map(
        location=[sum(lats)/len(lats), sum(longs)/len(longs)],
        zoom_start=15,
        tiles='cartodbpositron',
        width=width,
        height=height
    )

def visualize_hexagons(folium_map, hexagons, color="red"):
    """
    Отрисовка гексагонов на карте с использованием h3 v4.x
    hexagons: список или массив H3 индексов (строки)
    """
    for hex_cell in hexagons:
        if not hex_cell:
            continue
        try:
            boundary = h3.cell_to_boundary(hex_cell)
            polyline = [list(coord) for coord in boundary] + [list(boundary[0])]

            folium_map.add_child(folium.PolyLine(
                locations=polyline,
                weight=2,
                color=color,
                opacity=0.7
            ))
        except Exception as e:
            continue

    return folium_map

def visualize_stations(folium_map, stations, color="red"):
    """
    stations: DataFrame с колонками lat, long, name
    """
    for _, row in stations.iterrows():
        lat = row['lat']
        lng = row['long']
        name = row['name']

        if pd.isna(lat) or pd.isna(lng):
            continue

        folium_map.add_child(folium.Marker(
            location=(lat, lng),
            popup=name,
            icon=folium.Icon(color=color, icon='info-sign')
        ))

        folium_map.add_child(folium.Marker(
            location=(lat, lng),
            icon=folium.features.DivIcon(
                icon_size=(150, 36),
                icon_anchor=(0, 37),
                html=f'<div style="font-size: 8pt; white-space: nowrap">{name}</div>',
            ),
        ))
    return folium_map

In [105]:
m = init_map(h3_places.h3.unique())
visualize_hexagons(m, h3_places.h3.unique(), color="black")
visualize_stations(m, h3_places.loc[:, ['lat', 'long', 'name']])
display(m)

## Решение задач

1. Найти велосипед с максимальным временем пробега.
2. Найти наибольшее геодезическое расстояние между станциями.
3. Найти путь велосипеда с максимальным временем пробега через станции.
4. Найти количество велосипедов в системе.
5. Найти пользователей потративших на поездки более 3 часов.

### Найти велосипед с максимальным временем пробега

Суммируем общее время поездок для каждого велосипеда

In [106]:
bike_total_duration = tripData.groupBy('bike_id')\
    .agg(F.sum('duration').alias('total_duration_sec'))

bike_total_duration.show()

+-------+------------------+
|bike_id|total_duration_sec|
+-------+------------------+
|    471|           1718831|
|    496|           1679568|
|    148|            332138|
|    463|           1722796|
|    540|           1752835|
|    392|           1789476|
|    623|           2037219|
|    243|            307458|
|    516|           1896751|
|     31|            407907|
|    580|           1034382|
|    137|           1529200|
|    251|           1282980|
|    451|           1695574|
|     85|           1214769|
|    458|           1647080|
|     65|            216922|
|    588|            266415|
|    255|            396395|
|     53|            226389|
+-------+------------------+
only showing top 20 rows



Находим велосипед с максимальным суммарным временем

In [107]:
max_bike = bike_total_duration.orderBy(F.col('total_duration_sec').desc()).limit(1)
max_bike.show()

+-------+------------------+
|bike_id|total_duration_sec|
+-------+------------------+
|    535|          18611693|
+-------+------------------+



### Найти наибольшее геодезическое расстояние между станциями

UDF для расчёта геодезического расстояния

In [108]:
from geopy.distance import geodesic
from pyspark.sql.types import DoubleType

def calc_geodesic_distance(lat1, lon1, lat2, lon2):
    return geodesic((lat1, lon1), (lat2, lon2)).kilometers

geodesic_udf = F.udf(calc_geodesic_distance, DoubleType())

Создаём уникальные пары станций (исключаем дубли и пары с самой собой)

Используем условие s1.id < s2.id, чтобы каждая пара была только один раз

In [109]:
station_pairs = stationData.alias('s1').crossJoin(stationData.alias('s2'))\
    .filter(F.col('s1.id') < F.col('s2.id'))

Вычисляем расстояние для каждой пары

In [110]:
station_distances = station_pairs.withColumn(
    'distance_km',
    geodesic_udf(F.col('s1.lat'), F.col('s1.long'), F.col('s2.lat'), F.col('s2.long'))
).filter(F.col('distance_km').isNotNull())

Находим максимальное расстояние

In [111]:
max_distance = station_distances.orderBy(F.col('distance_km').desc())\
    .limit(1).select(
        F.col('s1.id').alias('station_1_id'),
        F.col('s2.id').alias('station_2_id'),
        F.col('distance_km')
    )

max_distance.show()

+------------+------------+-----------------+
|station_1_id|station_2_id|      distance_km|
+------------+------------+-----------------+
|          16|          60|69.92096757764355|
+------------+------------+-----------------+



### Найти путь велосипеда с максимальным временем пробега через станции

Получаем ID велосипеда с максимальным временем

In [112]:
max_bike_id = max_bike.first()['bike_id']
max_bike_id

535

Получаем все поездки этого велосипеда, отсортированные по времени

In [113]:
bike_path = tripData.filter(F.col('bike_id') == max_bike_id)\
    .orderBy('start_date')\
    .select(
        F.col('id'),
        F.col('start_station_id'),
        F.col('end_station_id'),
        F.col('start_station_name'),
        F.col('end_station_name')
    )
bike_path.show()

+-----+----------------+--------------+--------------------+--------------------+
|   id|start_station_id|end_station_id|  start_station_name|    end_station_name|
+-----+----------------+--------------+--------------------+--------------------+
| 4966|              47|            70|     Post at Kearney|San Francisco Cal...|
| 5067|              70|            69|San Francisco Cal...|San Francisco Cal...|
| 5179|              69|            77|San Francisco Cal...|   Market at Sansome|
| 5199|              77|            64|   Market at Sansome|   2nd at South Park|
| 7806|              61|            42|     2nd at Townsend|    Davis at Jackson|
|11422|              58|            72|San Francisco Cit...|Civic Center BART...|
|12245|              72|            47|Civic Center BART...|     Post at Kearney|
|12485|              47|            60|     Post at Kearney|Embarcadero at Sa...|
|12558|              60|            46|Embarcadero at Sa...|Washington at Kea...|
|13107|         

### Найти количество велосипедов в системе

In [114]:
bike_count = tripData.select('bike_id').distinct().count()
bike_count

700

### Найти пользователей потративших на поездки более 3 часов

In [115]:
users_over_3h = tripData.groupBy('zip_code')\
    .agg((F.sum('duration') / 3600).alias('total_hours'))\
    .filter(F.col('total_hours') > 3)\
    .orderBy(F.col('total_hours').desc())
users_over_3h.show()

+--------+------------------+
|zip_code|       total_hours|
+--------+------------------+
|   94107|13821.433888888889|
|     nil|12701.541666666666|
|    NULL| 7700.909166666666|
|   94105| 7110.035555555555|
|   94133| 6010.465277777777|
|   94102| 5313.339166666667|
|   94103| 5313.163333333333|
|   95531| 4797.333333333333|
|   94111|3956.9436111111113|
|   95112|3539.5472222222224|
|   94109| 3349.202222222222|
|   94040|2168.8683333333333|
|   94110| 2061.648888888889|
|   94117|1917.0313888888888|
|   94301|1830.6605555555554|
|   94041|1743.4122222222222|
|   94158|1735.6019444444444|
|   94306|1541.8452777777777|
|   94025|1438.3991666666666|
|   94108|1424.3227777777777|
+--------+------------------+
only showing top 20 rows

